<a href="https://colab.research.google.com/github/2303A52151/reinforcement_learning_2151/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Part A & B: Machine Initialization and Tables Setup
In this section, we set up our parameters, initialize our action counts ($N$), accumulated rewards ($R$), estimated values ($Q$), and UCB values for each of the three slot machines (A, B, and C).

In [1]:
import numpy as np
import pandas as pd

# Part A: Machine Initialization
machines = ['A', 'B', 'C']
M = len(machines)
N_play = 10
c = 2.0  # Exploration parameter for UCB (commonly set to 2.0)

# Initialize tables
N = {m: 0 for m in machines}      # Number of times each machine was played
R = {m: [] for m in machines}     # Rewards received from each machine
Q = {m: 0.0 for m in machines}    # Incremental sample-average Q-values
UCB = {m: float('inf') for m in machines} # UCB values (initialized to infinity to ensure initial exploration)

### Parts C, D & E: UCB Selection, Interactive Reward Collection, and Incremental Updates
Run this cell to begin the 10-play simulation. It will calculate the UCB for each machine, choose the one with the highest UCB, and ask you to input the corresponding reward based on your suggested reward pattern:
- **A**: 3, 4, 2, 5
- **B**: 6, 7, 5, 8
- **C**: 4, 3, 5, 4

In [4]:
total_reward_received = 0
history = []

print("Starting UCB Simulation for 10 Plays...\n")

for play in range(1, N_play + 1):
    # 1. Calculate UCB for each machine
    for m in machines:
        if N[m] == 0:
            UCB[m] = float('inf')  # Force selection of unplayed actions first
        else:
            # UCB Formula: Q(a) + c * sqrt(ln(t) / N(a))
            UCB[m] = Q[m] + c * np.sqrt(np.log(play) / N[m])

    # 2. Choose machine with the highest UCB value
    # Tie-breaking dynamically selects the first if there are multiples
    selected_machine = max(machines, key=lambda m: UCB[m])

    # 3. Ask user for reward
    print(f"Play {play}: Selected Machine is '{selected_machine}'")
    print(f"Current UCBs: " + ", ".join([f"{m}: {UCB[m]:.3f}" for m in machines]))

    try:
        reward = float(input(f"Enter reward obtained from machine {selected_machine}: "))
    except ValueError:
        print("Invalid input. Defaulting reward to 0.")
        reward = 0.0

    # 4. Update play counts and accumulated rewards
    N[selected_machine] += 1
    R[selected_machine].append(reward)
    total_reward_received += reward

    # 5. Update Q-value using incremental sample-average formula
    # Q_new = Q_old + (1/N) * (R - Q_old)
    old_q = Q[selected_machine]
    Q[selected_machine] = old_q + (1.0 / N[selected_machine]) * (reward - old_q)

    # Record state for analytical overview
    history.append({
        'Play': play,
        'Selected': selected_machine,
        'Reward': reward,
        'UCB_A': UCB['A'],
        'UCB_B': UCB['B'],
        'UCB_C': UCB['C'],
        'Q_A': Q['A'],
        'Q_B': Q['B'],
        'Q_C': Q['C']
    })
    print(f"Updated Q-value for {selected_machine}: {Q[selected_machine]:.3f}\n" + "-"*50)

print("Simulation Completed!")

Starting UCB Simulation for 10 Plays...

Play 1: Selected Machine is 'C'
Current UCBs: A: 33.000, B: 44.000, C: 225.000
Enter reward obtained from machine C: 33
Updated Q-value for C: 203.667
--------------------------------------------------
Play 2: Selected Machine is 'C'
Current UCBs: A: 34.665, B: 45.665, C: 204.222
Enter reward obtained from machine C: 33
Updated Q-value for C: 186.600
--------------------------------------------------
Play 3: Selected Machine is 'C'
Current UCBs: A: 35.096, B: 46.096, C: 187.263
Enter reward obtained from machine C: 33
Updated Q-value for C: 172.636
--------------------------------------------------
Play 4: Selected Machine is 'C'
Current UCBs: A: 35.355, B: 46.355, C: 173.346
Enter reward obtained from machine C: 33
Updated Q-value for C: 161.000
--------------------------------------------------
Play 5: Selected Machine is 'C'
Current UCBs: A: 35.537, B: 46.537, C: 161.732
Enter reward obtained from machine C: 33
Updated Q-value for C: 151.154


### Part F: Display Results & Calculations
This section displays the performance tables, logs the historical progress of the algorithm, and provides manual calculations for the first two steps to fulfill your submission criteria.

In [ ]:
# Display summary DataFrame
df_history = pd.DataFrame(history)
print("--- Historical Execution Log ---")
display(df_history)

print("\n--- Final Summary Table ---")
summary_data = {
    'Machine': machines,
    'Participation Count (N)': [N[m] for m in machines],
    'Total Reward (R_sum)': [sum(R[m]) for m in machines],
    'Final Estimated Q-value': [Q[m] for m in machines]
}
display(pd.DataFrame(summary_data))

print(f"\nTotal reward received by the player across all plays: {total_reward_received}")

### Theoretical Calculations for Submission (First 2 Plays)
Here is an automated step-by-step mathematical representation of the first 2 plays to include directly in your written submission:

#### **Play 1:**
* **UCB Calculation:** Since no machines have been played yet ($N(a) = 0$), their UCB values are initialized to $\infty$.
  * $UCB(A) = \infty$, $UCB(B) = \infty$, $UCB(C) = \infty$
* **Selection:** Machine **A** is selected (or whichever first tie-breaker occurs).
* **Reward:** User inputs $R = 3$.
* **Update:**
  * $N(A) = 1$
  * $Q(A) = 0 + \frac{1}{1}(3 - 0) = 3.0$

#### **Play 2:**
* **UCB Calculation:** (with $t = 2$)
  * $UCB(A) = Q(A) + c \times \sqrt{\frac{\ln(2)}{N(A)}} = 3.0 + 2.0 \times \sqrt{\frac{0.693}{1}} = 3.0 + 1.665 = 4.665$
  * $UCB(B) = \infty$ (since $N(B) = 0$)
  * $UCB(C) = \infty$ (since $N(C) = 0$)
* **Selection:** Machine **B** is selected (highest UCB value of $\infty$).
* **Reward:** User inputs $R = 6$.
* **Update:**
  * $N(B) = 1$
  * $Q(B) = 0 + \frac{1}{1}(6 - 0) = 6.0$